# 01 — Dataset exploration

**Goal:** understand how image files become PyTorch training batches.

In this notebook, you will inspect the CIFAR subset visually, build a working `ImageFolder` + `DataLoader`, and understand the batch tensor shape `[B, C, H, W]`.

After this notebook, open `data.py` and reproduce the same logic in the reusable script version.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 1. Check that the CIFAR subset exists

Before this notebook works, run this from the project root:

```bash
python prepare_cifar10_subset.py --train-per-class 400 --val-per-class 100
```

The folder should look like:

```text
data_cifar/
├── train/
│   ├── airplane/
│   ├── automobile/
│   ├── cat/
│   └── ship/
└── val/
    ├── airplane/
    ├── automobile/
    ├── cat/
    └── ship/
```


In [ ]:
DATA_DIR = PROJECT_ROOT / "data_cifar"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"

if not TRAIN_DIR.exists() or not VAL_DIR.exists():
    raise FileNotFoundError(
        "Could not find data_cifar/train and data_cifar/val. "
        "Run: python prepare_cifar10_subset.py --train-per-class 400 --val-per-class 100"
    )

print("Found dataset:", DATA_DIR)
print("Train folders:", sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]))
print("Val folders:", sorted([p.name for p in VAL_DIR.iterdir() if p.is_dir()]))


## 2. Build a working dataset in the notebook

`ImageFolder` is useful when images are arranged by class folders.

For example:

```text
data_cifar/train/cat/cat_0000.png  →  label = cat
```

This is the same idea you will later implement inside `data.py`.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import pandas as pd
import torch

IMAGE_SIZE = 32
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=transform)

class_names = train_dataset.classes
print("Class names:", class_names)
print("Number of training images:", len(train_dataset))
print("Number of validation images:", len(val_dataset))


## 3. Visualize class balance

For a controlled workshop experiment, the classes should be roughly balanced. If one class has many more examples, the model may learn to prefer that class.


In [ ]:
def count_by_class(dataset):
    counts = {name: 0 for name in dataset.classes}
    for _, label_idx in dataset.samples:
        counts[dataset.classes[label_idx]] += 1
    return counts

train_counts = count_by_class(train_dataset)
val_counts = count_by_class(val_dataset)

count_table = pd.DataFrame({"train": train_counts, "val": val_counts})
display(count_table)

ax = count_table.plot(kind="bar", figsize=(7, 4))
ax.set_title("Images per class")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Show sample images by class

This helps confirm that the folder names match the visual labels.


In [ ]:
def show_samples_by_class(dataset, class_names, samples_per_class=4):
    fig, axes = plt.subplots(len(class_names), samples_per_class, figsize=(2.2 * samples_per_class, 2.0 * len(class_names)))
    if len(class_names) == 1:
        axes = axes[None, :]

    for row, class_name in enumerate(class_names):
        class_idx = dataset.class_to_idx[class_name]
        indices = [i for i, (_, label) in enumerate(dataset.samples) if label == class_idx][:samples_per_class]
        for col, idx in enumerate(indices):
            image, label = dataset[idx]
            ax = axes[row, col]
            ax.imshow(image.permute(1, 2, 0))
            ax.set_title(class_name)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples_by_class(train_dataset, class_names)


## 5. Build a `DataLoader`

A `Dataset` lets us access one image at a time.

A `DataLoader` groups images into batches, optionally shuffles them, and gives batches to the model during training.


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image tensor dtype:", images.dtype)
print("Image value range:", float(images.min()), "to", float(images.max()))


## 6. What does `[B, C, H, W]` mean?

PyTorch image batches usually use this convention:

```text
B = batch size
C = channels
H = height
W = width
```

For our CIFAR batch:

```text
[B, C, H, W] = [64, 3, 32, 32]
```

This means:

```text
64 images per batch
3 color channels: red, green, blue
32 × 32 pixels per image
```

The label tensor has shape `[B]`, because each image has one class label.


In [ ]:
B, C, H, W = images.shape
print(f"B = {B} images per batch")
print(f"C = {C} channels")
print(f"H = {H} pixels tall")
print(f"W = {W} pixels wide")
print(f"labels.shape = {tuple(labels.shape)}")


## 7. Visualize one batch

Each image has a corresponding integer label. We convert that integer back to a class name using `class_names`.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
axes = axes.reshape(-1)

for i, ax in enumerate(axes):
    ax.imshow(images[i].permute(1, 2, 0))
    ax.set_title(f"label: {class_names[int(labels[i])]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## Student checkpoint: reproduce this in `data.py`

Open `data.py` and complete:

```text
TODO 1: build_transforms
TODO 2: create train_dataset and val_dataset using ImageFolder
TODO 3: create train_loader and val_loader using DataLoader
```

Then test your implementation from the terminal:

```bash
python train.py --config config.yaml --dry-run
```

Expected dry-run output:

```text
Input batch shape:   (64, 3, 32, 32)
Label batch shape:   (64,)
Output logits shape: (64, 4)
```

Questions to answer:

1. Why does the image batch have four dimensions?
2. Why does the label batch have one dimension?
3. Why should the training loader use `shuffle=True`?
